In [78]:
# ============================================================================
# Dephasing Matrix Extraction Tests for Context Operations
# ============================================================================
# This file contains comprehensive tests for the get_transformed_dephasing_matrix
# method, validating the extraction from superoperators and physical constraints.
# ============================================================================

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import mars
from mars import spin_model
from mars.population.contexts import Context, SummedContext
import mars.population.transform as transform

dtype = torch.float64


In [209]:
# ============================================================================
# 1. Helper Functions for Test Setup
# ============================================================================

def create_samples():
    """Create sample spin systems for testing."""
    g_tensor_1 = spin_model.Interaction((2.02, 2.04, 2.06), dtype=dtype)
    zfs_1 = spin_model.DEInteraction((200 * 1e6, 50 * 1e6), dtype=dtype)
    g_tensor_2 = spin_model.Interaction((2.02, 2.14, 2.16), dtype=dtype)
    zfs_2 = spin_model.DEInteraction((200 * 1e6, 70 * 1e6), dtype=dtype)
    
    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_1],
        electron_electron=[(0, 0, zfs_1)],
        dtype=dtype,
    )
    sample_1 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )

    base_spin_system = spin_model.SpinSystem(
        electrons=[1.0],
        g_tensors=[g_tensor_2],
        electron_electron=[(0, 0, zfs_2)],
        dtype=dtype
    )
    sample_2 = spin_model.MultiOrientedSample(
        base_spin_system=base_spin_system,
        gauss=0.001,
        lorentz=0.001,
        mesh=(4, 4),
        dtype=dtype,
    )
    return sample_1, sample_2


def get_eigen_basis(sample, field: float):
    """Compute eigenbasis for a sample at given magnetic field."""
    magnetic_field = torch.tensor(field)
    F, _, _, Gz = sample.get_hamiltonian_terms()
    H = F + Gz * magnetic_field
    H = H.unsqueeze(-3)
    values, vectors = torch.linalg.eigh(H)
    return vectors


def direct_sum_batched(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Compute the direct sum (block diagonal) of two batched square matrices."""
    n, n2 = A.shape[-2:]
    m, m2 = B.shape[-2:]
    
    assert n == n2, f"A must be square, got shape {A.shape}"
    assert m == m2, f"B must be square, got shape {B.shape}"
    
    batch_size = A.shape[:-2]
    total_dim = n + m
    
    result = torch.zeros(*batch_size, total_dim, total_dim, 
                         dtype=A.dtype, device=A.device)
    result[..., :n, :n] = A
    result[..., n:, n:] = B
    
    return result


def extract_dephasing_from_superop_manual(
    superop: torch.Tensor,
    dim: int,
    atol: float = 1e-10
) -> torch.Tensor:
    """
    Manually extract dephasing matrix from superoperator.
    
    The superoperator has shape [..., N², N²] where N is the Hilbert space dimension.
    In row-major (Liouville) ordering:
    - Index (i*N + j, i*N + j) corresponds to coherence ρ_ij diagonal element
    - For i ≠ j, this gives the dephasing rate γ_ij
    - For i = j, this gives population decay rates (set to 0 for dephasing matrix)
    
    Args:
        superop: Superoperator tensor of shape [..., N², N²]
        dim: Hilbert space dimension N
        atol: Tolerance for identifying coherence vs population blocks
    
    Returns:
        Dephasing matrix of shape [..., N, N]
    """
    return transform.extract_dephasing_matrix_from_superoperator(superop)


# ============================================================================
# 2. Context Creation Functions for Dephasing Tests
# ============================================================================

def create_contexts_dephasing_free_only(sample_1, sample_2, basis_1, basis_2):
    """Create contexts with only free_probs (dephasing from superoperator)."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        out_probs=[0.0, 0.0, 0.0],
        free_probs=torch.tensor([[-0.0, 100.0, 0.0], 
                                 [100.0, -0.0, 100.0], 
                                 [0.0, 100.0, -0.0]], dtype=dtype),
        dtype=dtype
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3], 
        out_probs=[0.0, 0.0, 0.0],
        free_probs=torch.tensor([[0.0, 100.0, 0.0], 
                                 [100.0, 0.0, 200.0], 
                                 [0.0, 200.0, -0.0]], dtype=dtype),
        dtype=dtype
    )
    return context_1, context_2


def create_contexts_dephasing_explicit(sample_1, sample_2, basis_1, basis_2):
    """Create contexts with explicit dephasing parameters."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.25, 0.25],
        out_probs=[1000.0, 200.3, 100.2],
        free_probs=torch.tensor([[-0.0, 100.0, 0.0], 
                                 [100.0, -0.0, 100.0], 
                                 [0.0, 100.0, -0.0]], dtype=dtype),
        dephasing=[1e4, 1e5, 1e6],
        dtype=dtype
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3], 
        out_probs=[111.0, 1234.0, 1232.0],
        free_probs=torch.tensor([[0.0, 100.0, 0.0], 
                                 [100.0, 0.0, 200.0], 
                                 [0.0, 200.0, -0.0]], dtype=dtype),
        dephasing=[2e5, 1e4, 1e3],
        dtype=dtype
    )
    return context_1, context_2


def create_contexts_dephasing_none(sample_1, sample_2, basis_1, basis_2):
    """Create contexts without any dephasing sources."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        out_probs=[0.0, 200.3, 100.2],
        free_probs=torch.tensor([[-0.0, 0.0, 0.0], 
                                 [0.0, -0.0, 0.0], 
                                 [0.0, 0.0, -0.0]], dtype=dtype) * 0.0,
        dtype=dtype
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3], 
        out_probs=[111.0, 1234.0, 1232.0],
        free_probs=torch.tensor([[0.0, 0.0, 0.0], 
                                 [0.0, 0.0, 0.0], 
                                 [0.0, 0.0, 0.0]], dtype=dtype) * 0.0,
        dtype=dtype
    )
    return context_1, context_2


def create_contexts_dephasing_driven_only(sample_1, sample_2, basis_1, basis_2):
    """Create contexts with driven_probs for dephasing from driven superoperator."""
    context_1 = Context(
        basis=basis_1,
        sample=sample_1,
        init_populations=[0.5, 0.35, 0.15],
        out_probs=[0.0, 200.3, 100.2],
        free_probs=torch.tensor([[-0.0, 50.0, 0.0], 
                                 [50.0, -0.0, 50.0], 
                                 [0.0, 50.0, -0.0]], dtype=dtype),
        driven_probs=torch.tensor([[0.0, 25.0, 0.0], 
                                   [25.0, 0.0, 25.0], 
                                   [0.0, 25.0, 0.0]], dtype=dtype),
        dtype=dtype
    )
    context_2 = Context(
        basis=basis_2,
        sample=sample_2,
        init_populations=[0.3, 0.4, 0.3], 
        out_probs=[0.0, 0.0, 0.0],
        free_probs=torch.tensor([[0.0, 50.0, 0.0], 
                                 [50.0, 0.0, 100.0], 
                                 [0.0, 100.0, -0.0]], dtype=dtype),
        driven_probs=torch.tensor([[0.0, 25.0, 0.0], 
                                   [25.0, 0.0, 50.0], 
                                   [0.0, 0.0, 0.0]], dtype=dtype),
        dtype=dtype
    )
    return context_1, context_2


def get_context_pairs(creator_func, sample_a, sample_b, use_same_sample: bool = False):
    """Generate context pairs for different basis combinations."""
    s1 = sample_a if use_same_sample else sample_a
    s2 = sample_a if use_same_sample else sample_b


    return [
        creator_func(s1, s2, "xyz", "zeeman"),
        creator_func(s1, s2, "zfs", "multiplet")
    ]


context_creators_dephasing = [
    create_contexts_dephasing_free_only,
    create_contexts_dephasing_explicit,
    create_contexts_dephasing_none,
    create_contexts_dephasing_driven_only,
]


def direct_sum_batched(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """Compute the direct sum (block diagonal) of two batched square matrices."""
    n, n2 = A.shape[-2:]
    m, m2 = B.shape[-2:]
    
    assert n == n2, f"A must be square, got shape {A.shape}"
    assert m == m2, f"B must be square, got shape {B.shape}"
    
    batch_size = A.shape[:-2]
    total_dim = n + m
    
    result = torch.zeros(*batch_size, total_dim, total_dim, 
                         dtype=A.dtype, device=A.device)
    result[..., :n, :n] = A
    result[..., n:, n:] = B
    
    return result


In [210]:
# ============================================================================
# 3. Dephasing Matrix Extraction Validation Functions
# ============================================================================

def check_dephasing_extraction_from_superop(
    context: Context,
    full_system_vectors: torch.Tensor,
    atol: float = 1e-6,
    rtol: float = 1e-5
) -> None:
    """
    Validate dephasing matrix extraction from free and driven superoperators.
    
    This test:
    1. Gets free and driven superoperators
    2. Sums them to get total superoperator
    3. Extracts diagonal elements from coherence blocks (row-major order)
    4. Compares with get_transformed_dephasing_matrix output
    """
    dim = context.spin_system_dim
    
    # Get superoperators
    superop_free = context.get_transformed_free_superop(full_system_vectors)
    superop_driven = context.get_transformed_driven_superop(full_system_vectors)
    
    # Get dephasing matrix from method
    dephasing_method = context.get_transformed_dephasing_matrix(full_system_vectors)

    # If no superoperators and no explicit dephasing, method should return None
    if superop_free is None and superop_driven is None:
        if context.dephasing is None:
            assert dephasing_method is None, "Expected None dephasing when no sources defined"
            print("  ✓ Dephasing extraction test passed: None as expected (no sources)")
            return
    
    # Sum superoperators if both exist
    if superop_free is not None and superop_driven is not None:
        superop_total = superop_free + superop_driven
    elif superop_free is not None:
        superop_total = superop_free
    elif superop_driven is not None:
        superop_total = superop_driven
    else:
        superop_total = None
    
    # If we have superoperator, extract dephasing manually
    print("-----------------------------------------------")
    dephasing_manual = extract_dephasing_from_superop_manual(superop_total, dim)
    print(dephasing_manual[0, 0] - dephasing_method[0, 0])

    diagonal = torch.diagonal(dephasing_method, dim1=-2, dim2=-1)
    assert torch.allclose(diagonal, torch.zeros_like(diagonal), atol=atol), \
            "Diagonal elements should be zero"
    assert torch.allclose(dephasing_method, dephasing_method.transpose(-2, -1), 
                             atol=atol, rtol=rtol), \
            "Dephasing matrix should be symmetric"
    assert torch.all(dephasing_method.real >= -1e-8), \
            f"Negative dephasing rates: min={dephasing_method.real.min().item():.2e}"

    assert torch.allclose(dephasing_method, dephasing_manual, atol=atol, rtol=rtol), \
        f"The max differnece is ={(dephasing_method - dephasing_manual).abs().max().item():.2e}"
    print("  ✓ Dephasing extraction test passed: physical constraints satisfied")

# ============================================================================
# 4. Comprehensive Dephasing Test Runner
# ============================================================================

def test_comprehensive_dephasing_operations(dtype=torch.float64):
    """Run all dephasing matrix tests."""
    sample_1, sample_2 = create_samples()
    vectors_1 = get_eigen_basis(sample_1, 5.1)
    vectors_2 = get_eigen_basis(sample_2, 5.1)
    
    print("=" * 70)
    print("Testing Dephasing Matrix Operations")
    print("=" * 70)
    
    # Test 1: Dephasing extraction from free/driven superoperators
    print("\n1. Testing dephasing extraction from superoperators...")
    for creator in context_creators_dephasing:
        for ctx1, ctx2 in get_context_pairs(creator, sample_1, sample_2, use_same_sample=False):
            check_dephasing_extraction_from_superop(ctx1, vectors_1)
            check_dephasing_extraction_from_superop(ctx2, vectors_2)
            
    
    # Test 2: Dephasing with multiplication
    print("\n3. Testing dephasing  under multiplication...")
    for creator in context_creators_dephasing:
        for ctx1, ctx2 in get_context_pairs(creator, sample_1, sample_2, use_same_sample=False):
            mul_context = ctx1 @ ctx2
            mul_vectors = transform.batched_kron(vectors_1.clone(), vectors_2.clone())
            check_dephasing_extraction_from_superop(mul_context, mul_vectors)
    
    # Test 3: Dephasing with concatinations
    print("\n4. Testing dephasing under concatination...")
    for creator in context_creators_dephasing:
        for ctx1, ctx2 in get_context_pairs(creator, sample_1, sample_2, use_same_sample=False):
            concat_context = mars.concat([ctx1, ctx2])
            concat_vectors = direct_sum_batched(vectors_1, vectors_2)
            
            check_dephasing_extraction_from_superop(mul_context, mul_vectors)
    
    print("\n" + "=" * 70)
    print("✅ All dephasing matrix tests passed successfully!")
    print("=" * 70)

In [211]:
if __name__ == "__main__":
    test_comprehensive_dephasing_operations()

Testing Dephasing Matrix Operations

1. Testing dephasing extraction from superoperators...
-----------------------------------------------
tensor([[-0.0000e+00, 1.4211e-14, 0.0000e+00],
        [1.4211e-14, -0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, -0.0000e+00]], dtype=torch.float64)
  ✓ Dephasing extraction test passed: physical constraints satisfied
-----------------------------------------------
tensor([[-0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00, -0.0000e+00, -2.8422e-14],
        [ 0.0000e+00,  0.0000e+00, -0.0000e+00]], dtype=torch.float64)
  ✓ Dephasing extraction test passed: physical constraints satisfied
-----------------------------------------------
tensor([[-0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, -0.0000e+00, 1.4211e-14],
        [2.8422e-14, 2.8422e-14, -0.0000e+00]], dtype=torch.float64)
  ✓ Dephasing extraction test passed: physical constraints satisfied
-----------------------------------------------
tensor([[-0.00

RuntimeError: shape '[9, 9, 9]' is invalid for input of size 81